# 01a. 채널 정보 수집 — `channels.list` API

**목적:** youtube_channels.csv 의 head(5) 샘플 채널에 대해  
YouTube Data API v3 `channels.list` 로 받을 수 있는 **모든 필드**를 확인한다.

| 표시 | 의미 |
|------|------|
| ⭐ MUST | 이탈 예측 피처 또는 영상 수집에 필수 |
| ○ OPTIONAL | 있으면 좋지만 없어도 됨 |

**API Quota 비용:** `channels.list` 1회 = **1 unit** (50채널 배치 가능)

## 0. 설정

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

API_KEY = os.getenv('YOUTUBE_API_KEY')
assert API_KEY, '.env 파일에 YOUTUBE_API_KEY 를 설정하세요'

youtube = build('youtube', 'v3', developerKey=API_KEY)

ROOT     = Path('../../')          # 프로젝트 루트
CSV_PATH = ROOT / 'data' / 'raw' / 'youtube_channels.csv'
OUT_DIR  = ROOT / 'data' / 'raw' / 'channels'
JSON_DIR = OUT_DIR / 'json'        # 원시 API 응답 저장
CSV_DIR  = OUT_DIR / 'csv'         # MUST 필드 가공본
JSON_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 처리할 youtube_channels.csv 의 행 인덱스 범위 (둘 다 inclusive)
# 예: START_IDX=0, END_IDX=49 → 0번째 ~ 49번째 행 (총 50개 채널) 처리
# 다음 실행 때는 직전 실행이 출력한 안내 메시지의 인덱스부터 이어서 처리
START_IDX = 0
END_IDX   = 49
# ─────────────────────────────────────────────────────────────

print('API 연결 완료')

## 1. 처리 대상 채널 불러오기 (`START_IDX` ~ `END_IDX`)

In [ ]:
df_all = pd.read_csv(CSV_PATH)
df = df_all.iloc[START_IDX:END_IDX + 1].copy()

print(f'전체 채널 수: {len(df_all):,}')
print(f'이번 실행 범위: {START_IDX} ~ {END_IDX} (총 {len(df)}개 채널)')
df[['channel_name', 'youtube_channel_id']].head(10)

In [3]:
channel_ids = df['youtube_channel_id'].tolist()
print('수집할 channel_id:')
for cid in channel_ids:
    print(' ', cid)

수집할 channel_id:
  UCo3Yj54VtkEvQX9cLHKklzw
  UCyABUa7lzjsV4o5PfSFqymw
  UCGPHpVMxzO2rc5SqhGIc2tg
  UCDfRXnMKZmokf5Jtl8K6nHQ
  UCra9lyIlp4Q3eE-cDiEiJbw


## 2. `channels.list` — 받을 수 있는 모든 part 호출

| Part | 주요 내용 |
|------|-----------|
| `snippet` | 채널명·설명·개설일·국가·썸네일 |
| `contentDetails` | ⭐ uploads 플레이리스트 ID (영상 수집 필수) |
| `statistics` | ⭐ 구독자 수·총 조회수·총 영상 수 |
| `topicDetails` | 콘텐츠 카테고리 토픽 |
| `brandingSettings` | 채널 키워드·배너 |
| `status` | 공개 상태·성인 인증 여부 |

In [ ]:
PARTS = 'snippet,contentDetails,statistics,topicDetails,brandingSettings,status'
BATCH_SIZE = 50   # channels.list API 한 번에 최대 50개 ID

QUOTA_REASONS = {'quotaExceeded', 'rateLimitExceeded', 'dailyLimitExceeded', 'userRateLimitExceeded'}

channel_ids_all = df['youtube_channel_id'].tolist()
all_items = []                       # 누적된 channels.list 응답 items
last_completed_idx = START_IDX - 1   # 마지막으로 성공한 df_all 기준 인덱스
api_error = None                     # quota 등 에러 메시지

for batch_start in range(0, len(channel_ids_all), BATCH_SIZE):
    batch_ids   = channel_ids_all[batch_start:batch_start + BATCH_SIZE]
    range_lo    = START_IDX + batch_start
    range_hi    = START_IDX + batch_start + len(batch_ids) - 1
    print(f'  batch {batch_start // BATCH_SIZE + 1}: df_all[{range_lo}..{range_hi}] ({len(batch_ids)}개)')
    try:
        resp = youtube.channels().list(
            part=PARTS,
            id=','.join(batch_ids),
            maxResults=BATCH_SIZE,
        ).execute()
        all_items.extend(resp.get('items', []))
        last_completed_idx = range_hi
    except HttpError as e:
        reason = e.error_details[0]['reason'] if e.error_details else f'HTTP {e.resp.status}'
        api_error = reason
        print(f'  ✗ API 에러: {reason} — 이 배치 이후 중단')
        if reason not in QUOTA_REASONS:
            print('  (quota 에러는 아니지만 안전을 위해 중단합니다)')
        break

response = {'items': all_items}   # 이후 셀에서 사용하는 변수와 호환
n_units  = (last_completed_idx - START_IDX + 1 + BATCH_SIZE - 1) // BATCH_SIZE if last_completed_idx >= START_IDX else 0

print(f'\n수집 결과 — 응답 {len(all_items)}개 / 사용 quota: 약 {n_units} unit')
print(f'성공한 마지막 인덱스: {last_completed_idx}')
if api_error:
    print(f'에러 발생: {api_error}')

## 3. 전체 응답 구조 확인

In [5]:
# 첫 번째 채널의 전체 응답을 그대로 출력
sample = response['items'][0]
print(json.dumps(sample, indent=2, ensure_ascii=False))

{
  "kind": "youtube#channel",
  "etag": "rAUEqez1as2xk-yPpQHyuctfz8E",
  "id": "UCra9lyIlp4Q3eE-cDiEiJbw",
  "snippet": {
    "title": " - Topic",
    "description": "",
    "publishedAt": "2018-05-15T13:25:55Z",
    "thumbnails": {
      "default": {
        "url": "https://yt3.ggpht.com/Adm3zsEy0ce22zRNS8gtKY0egTJ_3y27JMLnC9ty3XtHdfhEhu7an5H4xRv3BgbM96A0yohqRg=s88-c-k-c0x00ffffff-no-rj",
        "width": 88,
        "height": 88
      },
      "medium": {
        "url": "https://yt3.ggpht.com/Adm3zsEy0ce22zRNS8gtKY0egTJ_3y27JMLnC9ty3XtHdfhEhu7an5H4xRv3BgbM96A0yohqRg=s240-c-k-c0x00ffffff-no-rj",
        "width": 240,
        "height": 240
      },
      "high": {
        "url": "https://yt3.ggpht.com/Adm3zsEy0ce22zRNS8gtKY0egTJ_3y27JMLnC9ty3XtHdfhEhu7an5H4xRv3BgbM96A0yohqRg=s800-c-k-c0x00ffffff-no-rj",
        "width": 800,
        "height": 800
      }
    },
    "localized": {
      "title": " - Topic",
      "description": ""
    }
  },
  "contentDetails": {
    "relatedPlaylists"

## 4. 필드별 상세 확인

### 4-1. snippet (기본 정보)

In [6]:
rows = []
for item in response['items']:
    s = item.get('snippet', {})
    rows.append({
        # ⭐ MUST — 채널 식별
        'channel_id':        item['id'],
        'title':             s.get('title'),            # ⭐ MUST  채널명
        'published_at':      s.get('publishedAt'),       # ⭐ MUST  채널 개설일 → 채널 나이 피처
        # ○ OPTIONAL
        'description':       s.get('description'),       # ○ 채널 설명
        'custom_url':        s.get('customUrl'),          # ○ @핸들
        'country':           s.get('country'),            # ○ 국가 코드
        'default_language':  s.get('defaultLanguage'),   # ○ 기본 언어
        'thumbnail_default': s.get('thumbnails', {}).get('default', {}).get('url'),
        'thumbnail_high':    s.get('thumbnails', {}).get('high', {}).get('url'),
    })

pd.DataFrame(rows)

,channel_id,title,published_at,description,custom_url,country,default_language,thumbnail_default,thumbnail_high
0,UCra9lyIlp4Q3eE-cDiEiJbw,- Topic,2018-05-15T13:25:55Z,,None,None,None,https://yt3.ggpht.com/Adm3zsEy0ce22zRNS8gtKY0e...,https://yt3.ggpht.com/Adm3zsEy0ce22zRNS8gtKY0e...
1,UCGPHpVMxzO2rc5SqhGIc2tg,의원실김정우,2016-06-04T13:29:52Z,,@의원실김정우,None,None,https://yt3.ggpht.com/ytc/AIdro_kWETz648WkDdPy...,https://yt3.ggpht.com/ytc/AIdro_kWETz648WkDdPy...
2,UCyABUa7lzjsV4o5PfSFqymw,심해 생존일지스타트의,2017-08-06T17:58:34Z,명예로운 삶을!하!영광스러운 죽음을!,@심해생존일지스타트의,None,None,https://yt3.ggpht.com/ytc/AIdro_mSiY7GIAmasjvo...,https://yt3.ggpht.com/ytc/AIdro_mSiY7GIAmasjvo...
3,UCo3Yj54VtkEvQX9cLHKklzw,장정숙,2016-06-08T06:54:31Z,,@장정숙-r7l,None,None,https://yt3.ggpht.com/ytc/AIdro_lNQJUkt6-zJNPQ...,https://yt3.ggpht.com/ytc/AIdro_lNQJUkt6-zJNPQ...
4,UCDfRXnMKZmokf5Jtl8K6nHQ,장석주,2013-10-08T07:03:43Z,,@장석주-t4v,KR,None,https://yt3.ggpht.com/ytc/AIdro_mGXAcZ2AzNxLTF...,https://yt3.ggpht.com/ytc/AIdro_mGXAcZ2AzNxLTF...


### 4-2. contentDetails — ⭐ uploads 플레이리스트 ID (영상 수집 핵심)

In [7]:
rows = []
for item in response['items']:
    cd = item.get('contentDetails', {}).get('relatedPlaylists', {})
    rows.append({
        'channel_id':       item['id'],
        'uploads_playlist': cd.get('uploads'),   # ⭐ MUST  영상 목록 수집에 필수
        'likes_playlist':   cd.get('likes'),      # ○ 좋아요 재생목록
    })

df_content = pd.DataFrame(rows)
df_content

,channel_id,uploads_playlist,likes_playlist
0,UCra9lyIlp4Q3eE-cDiEiJbw,UUra9lyIlp4Q3eE-cDiEiJbw,
1,UCGPHpVMxzO2rc5SqhGIc2tg,UUGPHpVMxzO2rc5SqhGIc2tg,
2,UCyABUa7lzjsV4o5PfSFqymw,UUyABUa7lzjsV4o5PfSFqymw,
3,UCo3Yj54VtkEvQX9cLHKklzw,UUo3Yj54VtkEvQX9cLHKklzw,
4,UCDfRXnMKZmokf5Jtl8K6nHQ,UUDfRXnMKZmokf5Jtl8K6nHQ,


### 4-3. statistics — ⭐ 구독자·조회수·영상 수

In [8]:
rows = []
for item in response['items']:
    st = item.get('statistics', {})
    rows.append({
        'channel_id':              item['id'],
        'subscriber_count':        st.get('subscriberCount'),         # ⭐ MUST  구독자 수
        'view_count':              st.get('viewCount'),                # ⭐ MUST  총 채널 조회수
        'video_count':             st.get('videoCount'),               # ⭐ MUST  총 영상 수
        'hidden_subscriber_count': st.get('hiddenSubscriberCount'),   # ○ 구독자 수 비공개 여부
    })

pd.DataFrame(rows)

,channel_id,subscriber_count,view_count,video_count,hidden_subscriber_count
0,UCra9lyIlp4Q3eE-cDiEiJbw,1,0,0,False
1,UCGPHpVMxzO2rc5SqhGIc2tg,1,41,1,False
2,UCyABUa7lzjsV4o5PfSFqymw,1,942,3,False
3,UCo3Yj54VtkEvQX9cLHKklzw,1,493,3,False
4,UCDfRXnMKZmokf5Jtl8K6nHQ,1,0,0,False


### 4-4. topicDetails (카테고리)

In [9]:
rows = []
for item in response['items']:
    td = item.get('topicDetails', {})
    rows.append({
        'channel_id':       item['id'],
        'topic_ids':        td.get('topicIds'),           # ○ Freebase 토픽 ID 목록
        'topic_categories': td.get('topicCategories'),   # ○ Wikipedia 카테고리 URL
    })

pd.DataFrame(rows)

,channel_id,topic_ids,topic_categories
0,UCra9lyIlp4Q3eE-cDiEiJbw,None,None
1,UCGPHpVMxzO2rc5SqhGIc2tg,None,None
2,UCyABUa7lzjsV4o5PfSFqymw,"[/m/0bzvm2, /m/025zzc]",[https://en.wikipedia.org/wiki/Video_game_cult...
3,UCo3Yj54VtkEvQX9cLHKklzw,"[/m/05qt0, /m/098wr]","[https://en.wikipedia.org/wiki/Politics, https..."
4,UCDfRXnMKZmokf5Jtl8K6nHQ,None,None


### 4-5. brandingSettings

In [10]:
rows = []
for item in response['items']:
    ch = item.get('brandingSettings', {}).get('channel', {})
    img = item.get('brandingSettings', {}).get('image', {})
    rows.append({
        'channel_id':             item['id'],
        'keywords':               ch.get('keywords'),               # ○ 채널 키워드 (태그)
        'unsubscribed_trailer':   ch.get('unsubscribedTrailer'),    # ○ 구독 전 트레일러 영상 ID
        'banner_url':             img.get('bannerExternalUrl'),      # ○ 채널 배너 이미지 URL
    })

pd.DataFrame(rows)

,channel_id,keywords,unsubscribed_trailer,banner_url
0,UCra9lyIlp4Q3eE-cDiEiJbw,None,None,https://yt3.googleusercontent.com/zumwEMTxpLdK...
1,UCGPHpVMxzO2rc5SqhGIc2tg,None,None,None
2,UCyABUa7lzjsV4o5PfSFqymw,None,None,https://yt3.googleusercontent.com/NcUQ84rlXiH6...
3,UCo3Yj54VtkEvQX9cLHKklzw,None,None,None
4,UCDfRXnMKZmokf5Jtl8K6nHQ,None,None,None


### 4-6. status

In [11]:
rows = []
for item in response['items']:
    s = item.get('status', {})
    rows.append({
        'channel_id':          item['id'],
        'privacy_status':      s.get('privacyStatus'),       # ○ public / private / unlisted
        'is_linked':           s.get('isLinked'),             # ○ Google 계정 연결 여부
        'long_uploads_status': s.get('longUploadsStatus'),   # ○ 15분 초과 업로드 가능 여부
        'made_for_kids':       s.get('madeForKids'),         # ○ 어린이용 채널 여부
    })

pd.DataFrame(rows)

,channel_id,privacy_status,is_linked,long_uploads_status,made_for_kids
0,UCra9lyIlp4Q3eE-cDiEiJbw,public,True,longUploadsUnspecified,None
1,UCGPHpVMxzO2rc5SqhGIc2tg,public,True,longUploadsUnspecified,None
2,UCyABUa7lzjsV4o5PfSFqymw,public,True,longUploadsUnspecified,None
3,UCo3Yj54VtkEvQX9cLHKklzw,public,True,longUploadsUnspecified,None
4,UCDfRXnMKZmokf5Jtl8K6nHQ,public,True,longUploadsUnspecified,None


## 5. 이탈 예측에 사용할 필드 정리

| 필드 | Part | 용도 |
|------|------|------|
| `channel_id` | — | 키 |
| `published_at` | snippet | ⭐ 채널 나이 계산 |
| `country` | snippet | ○ 국가 피처 |
| `uploads_playlist` | contentDetails | ⭐ 영상 수집 (01b 노트북) |
| `subscriber_count` | statistics | ⭐ 규모 피처 |
| `view_count` | statistics | ⭐ 총 조회수 |
| `video_count` | statistics | ⭐ 총 영상 수 |

## 6. 결과 저장

In [ ]:
# 실제로 성공한 범위
actual_end_idx = last_completed_idx
total = len(df_all)

if actual_end_idx < START_IDX or not response['items']:
    print('⚠ 한 건도 수집하지 못함 — CSV 저장을 생략합니다.')
    print(f'▶ 다음 실행 시 START_IDX = {START_IDX} 로 다시 시도하세요.')
else:
    # 원시 API 응답 전체 저장
    out_raw = JSON_DIR / f'channels_raw_{START_IDX}_{actual_end_idx}.json'
    with open(out_raw, 'w', encoding='utf-8') as f:
        json.dump(response['items'], f, indent=2, ensure_ascii=False)
    print(f'원시 저장: {out_raw}')

    # MUST 필드 추출
    must_rows = []
    for item in response['items']:
        s  = item.get('snippet', {})
        cd = item.get('contentDetails', {}).get('relatedPlaylists', {})
        st = item.get('statistics', {})
        must_rows.append({
            'channel_id':       item['id'],
            'title':            s.get('title'),
            'published_at':     s.get('publishedAt'),
            'country':          s.get('country'),
            'uploads_playlist': cd.get('uploads'),
            'subscriber_count': st.get('subscriberCount'),
            'view_count':       st.get('viewCount'),
            'video_count':      st.get('videoCount'),
        })

    df_must = pd.DataFrame(must_rows)
    out_csv = CSV_DIR / f'channels_must_{START_IDX}_{actual_end_idx}.csv'
    df_must.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f'MUST 필드 저장: {out_csv} ({len(df_must)}개 행)')

    # 다음 실행 안내
    if actual_end_idx < END_IDX:
        print(f'\n⚠ API 에러로 일부만 처리됨 — 요청 {START_IDX}~{END_IDX} / 성공 {START_IDX}~{actual_end_idx}')
        print(f'▶ 다음 실행 시 START_IDX = {actual_end_idx + 1} 부터 다시 시작하세요.')
    else:
        print(f'\n✓ 요청 범위 {START_IDX}~{END_IDX} 전체 처리 완료')
        if actual_end_idx + 1 < total:
            remaining = total - actual_end_idx - 1
            print(f'▶ 다음 실행 시 START_IDX = {actual_end_idx + 1} (전체 {total}개 중 {remaining}개 남음)')
        else:
            print(f'▶ youtube_channels.csv 전체 처리 완료 (총 {total}개)')

    df_must.head()